In [8]:
# ----------------------------------------------------------
# DOCUMENTATION NOTE
# ----------------------------------------------------------
# This integration step demonstrates multi-source data engineering:
#
# 1. Relational source data:
#    - customers
#    - orders
#    - payments
#
# 2. External API enrichment:
#    - products
#
# 3. Data harmonization:
#    - standardized product_id before joining
#
# 4. Integrated Silver output:
#    - one unified analytics-ready dataset
#
# Real-world value:
# This simulates combining transactional data with external enrichment data
# to build a business-consumable analytics layer.

print("Multi-source integration logic documented successfully.")

StatementMeta(, 3b10e3b8-907e-41b3-bc5f-bd9d73e44ba5, 10, Finished, Available, Finished, False)

Multi-source integration logic documented successfully.


In [2]:
# STEP 1: Read source tables for integration

df_customers = spark.read.table("bronze_sql_customers")
df_orders = spark.read.table("bronze_sql_orders")
df_payments = spark.read.table("bronze_sql_payments")
df_products = spark.read.table("silver_api_products_clean")

print("Orders sample:")
df_orders.select("order_id", "product_id").show(truncate=False)

print("Products sample:")
df_products.select("product_id", "product_name", "category").show(10, truncate=False)

StatementMeta(, 3b10e3b8-907e-41b3-bc5f-bd9d73e44ba5, 4, Finished, Available, Finished, False)

Orders sample:
+--------+----------+
|order_id|product_id|
+--------+----------+
|9001    |3001      |
|9003    |3003      |
|9002    |3002      |
+--------+----------+

Products sample:
+----------+-----------------------------+----------+
|product_id|product_name                 |category  |
+----------+-----------------------------+----------+
|1         |Essence Mascara Lash Princess|beauty    |
|2         |Eyeshadow Palette with Mirror|beauty    |
|3         |Powder Canister              |beauty    |
|7         |Chanel Coco Noir Eau De      |fragrances|
|8         |Dior J'adore                 |fragrances|
|9         |Dolce Shine Eau de           |fragrances|
|4         |Red Lipstick                 |beauty    |
|5         |Red Nail Polish              |beauty    |
|6         |Calvin Klein CK One          |fragrances|
|16        |Apple                        |groceries |
+----------+-----------------------------+----------+
only showing top 10 rows



In [3]:
from pyspark.sql.functions import when, col

# ----------------------------------------------------------
# STEP 2: STANDARDIZE PRODUCT IDs ACROSS SYSTEMS
# ----------------------------------------------------------
# Problem:
# Orders source table uses product_id values like 3001, 3002, 3003
# API product table uses product_id values like 1, 2, 3
#
# To make the join work, we remap the order product IDs so they match
# the external API product IDs.
#
# Real-world concept:
# This simulates "key standardization" across multiple systems.

df_orders_fixed = df_orders.withColumn(
    "product_id",
    when(col("product_id") == 3001, 1)   # map source product 3001 -> API product 1
    .when(col("product_id") == 3002, 2)  # map source product 3002 -> API product 2
    .when(col("product_id") == 3003, 3)  # map source product 3003 -> API product 3
    .otherwise(col("product_id"))        # keep original value if not part of mapping
)

# Validate that the product_id values are now aligned for joining
df_orders_fixed.show(truncate=False)

StatementMeta(, 3b10e3b8-907e-41b3-bc5f-bd9d73e44ba5, 5, Finished, Available, Finished, False)

+--------+-----------+----------+-------------------+------------+------------+
|order_id|customer_id|product_id|order_time         |order_amount|order_status|
+--------+-----------+----------+-------------------+------------+------------+
|9001    |101        |1         |2026-04-13 15:30:00|120.5       |CREATED     |
|9003    |103        |3         |2026-04-13 15:32:00|220.0       |CREATED     |
|9002    |102        |2         |2026-04-13 15:31:00|80.0        |CREATED     |
+--------+-----------+----------+-------------------+------------+------------+



In [4]:
# ----------------------------------------------------------
# STEP 3: MULTI-SOURCE INTEGRATION JOIN
# ----------------------------------------------------------
# We now combine:
# - orders from relational source
# - customers from relational source
# - payments from relational source
# - products from external API source
#
# Join strategy:
# orders is the main transactional table
# customer_id links orders -> customers
# order_id links orders -> payments
# product_id links orders -> products

df_integrated = (
    df_orders_fixed.alias("o")
    .join(df_customers.alias("c"), on="customer_id", how="left")  # enrich orders with customer details
    .join(df_payments.alias("p"), on="order_id", how="left")      # enrich orders with payment details
    .join(df_products.alias("pr"), on="product_id", how="left")   # enrich orders with product/API details
)

# Preview joined data before final column selection
df_integrated.show(truncate=False)

StatementMeta(, 3b10e3b8-907e-41b3-bc5f-bd9d73e44ba5, 6, Finished, Available, Finished, False)

+----------+--------+-----------+-------------------+------------+------------+-------------+------------------+---------+----------+-------------------+--------------+--------------+--------------+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------------+-----+-------------------+------+-----+-------------------+---------------+------+---------------+----------------+---------------+--------------------------+--------------------+----------------------+------------------------+------------------------+--------------------------------------------------------------------------------------------+
|product_id|order_id|customer_id|order_time         |order_amount|order_status|customer_name|email             |city     |payment_id|payment_time       |payment_amount|payment_status|payment_method|product_name      

In [5]:
# ----------------------------------------------------------
# STEP 4: SELECT CLEAN BUSINESS COLUMNS
# ----------------------------------------------------------
# The joined DataFrame contains many columns from multiple sources.
# We now pick only the columns needed for downstream analytics.

df_final = df_integrated.select(
    col("o.order_id").alias("order_id"),
    col("o.customer_id").alias("customer_id"),
    col("c.customer_name").alias("customer_name"),
    col("c.email").alias("email"),
    col("c.city").alias("city"),
    col("o.product_id").alias("product_id"),
    col("pr.product_name").alias("product_name"),
    col("pr.category").alias("category"),
    col("pr.brand").alias("brand"),
    col("o.order_time").alias("order_time"),
    col("o.order_amount").alias("order_amount"),
    col("o.order_status").alias("order_status"),
    col("p.payment_id").alias("payment_id"),
    col("p.payment_time").alias("payment_time"),
    col("p.payment_amount").alias("payment_amount"),
    col("p.payment_status").alias("payment_status"),
    col("p.payment_method").alias("payment_method")
)

# Preview final integrated Silver-ready dataset
df_final.show(truncate=False)

StatementMeta(, 3b10e3b8-907e-41b3-bc5f-bd9d73e44ba5, 7, Finished, Available, Finished, False)

+--------+-----------+-------------+------------------+---------+----------+-----------------------------+--------+--------------+-------------------+------------+------------+----------+-------------------+--------------+--------------+--------------+
|order_id|customer_id|customer_name|email             |city     |product_id|product_name                 |category|brand         |order_time         |order_amount|order_status|payment_id|payment_time       |payment_amount|payment_status|payment_method|
+--------+-----------+-------------+------------------+---------+----------+-----------------------------+--------+--------------+-------------------+------------+------------+----------+-------------------+--------------+--------------+--------------+
|9001    |101        |Santhosh     |santhosh@gmail.com|Bangalore|1         |Essence Mascara Lash Princess|beauty  |Essence       |2026-04-13 15:30:00|120.5       |CREATED     |5001      |2026-04-13 15:33:00|120.5         |SUCCESS       |card

In [6]:
# ----------------------------------------------------------
# STEP 5: WRITE TO SILVER LAYER
# ----------------------------------------------------------
# This creates the integrated Silver table that will be used
# for analytics, business rules, and Gold layer reporting.

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders_integrated")

print("silver_orders_integrated table created successfully.")

StatementMeta(, 3b10e3b8-907e-41b3-bc5f-bd9d73e44ba5, 8, Finished, Available, Finished, False)

silver_orders_integrated table created successfully.


In [7]:
# ----------------------------------------------------------
# STEP 6: VALIDATE SILVER OUTPUT
# ----------------------------------------------------------
# Read the final Silver table and verify the integrated records.

spark.sql("""
SELECT *
FROM silver_orders_integrated
ORDER BY order_id
""").show(truncate=False)

StatementMeta(, 3b10e3b8-907e-41b3-bc5f-bd9d73e44ba5, 9, Finished, Available, Finished, False)

+--------+-----------+-------------+------------------+---------+----------+-----------------------------+--------+--------------+-------------------+------------+------------+----------+-------------------+--------------+--------------+--------------+
|order_id|customer_id|customer_name|email             |city     |product_id|product_name                 |category|brand         |order_time         |order_amount|order_status|payment_id|payment_time       |payment_amount|payment_status|payment_method|
+--------+-----------+-------------+------------------+---------+----------+-----------------------------+--------+--------------+-------------------+------------+------------+----------+-------------------+--------------+--------------+--------------+
|9001    |101        |Santhosh     |santhosh@gmail.com|Bangalore|1         |Essence Mascara Lash Princess|beauty  |Essence       |2026-04-13 15:30:00|120.5       |CREATED     |5001      |2026-04-13 15:33:00|120.5         |SUCCESS       |card